###Ingest CBS csv file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe write api

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/cbs/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.cbs"

In [0]:
from pyspark.sql.types import *

In [0]:
cbs_schema = StructType([
  StructField('TransactionID', StringType(), True),
  StructField('BillerID', StringType(), True),
  StructField('CustomerID', StringType(), True),
  StructField('CreditAmount', DoubleType(), True),
  StructField('CreditStatus', StringType(), True),
  StructField('BankReferenceNo', StringType(), True),
  StructField('CreditTimestamp', DateType(), True)
])

In [0]:
cbs_df = spark.read.format('csv') \
                .option('header', True) \
                .schema(cbs_schema) \
                .option('mode', 'FAILFAST') \
                .load(source_file)

In [0]:
cbs_audit = add_ingestion_metadata(cbs_df)

In [0]:
cbs_final= cbs_audit.withColumn("batch_id", F.lit(v_batch_id))

In [0]:
cbs_final.write.mode('overwrite').partitionBy('batch_id').option('replaceWhere',f"batch_id = '{v_batch_id}'").saveAsTable(table_name)